# Sim-to-real benchmark of ECG image generators

Trains the **Open-ECG-Digitizer** segmentation network (npj Digital Medicine, pinned in `reproductions/npj`) with the *same budget* on images from each generator recipe, digitizes the *same real* ECG-Image-Database photos and scans with the native npj pipeline, and scores the signals against PTB-XL references. Only the training images change between arms.

**Arms**
- `published`: the authors' released weights, trained on their own full data. A reference point, not an equal-budget arm.
- `legacy`: this renderer with default settings.
- `full`: this renderer with every realism option from 0.14–0.31 enabled.
- Optional ablations `full-no-<flag>`, for example `full-no-phone_isp`.

**Before running:** *Runtime → Change runtime type → GPU* (T4 works; A100/L4 is faster). Results are written to Google Drive as each arm finishes, so an interrupted session keeps what was done.

**Rough time on a T4 with the defaults:** downloads 10–15 min, rendering 15–25 min per arm, training 40–70 min per arm, inference 10–20 min per arm. Lower `STEPS` / `TRAIN_RECORDS` for a quick pilot.

**Data licences.** PTB-XL (CC BY 4.0) and ECG-Image-Database (CC BY-ND 4.0) are downloaded into this Colab session only. Do not redistribute images.

In [ ]:
# ---- Configuration ---------------------------------------------------------
import time
REPO_REF = 'main'                       # branch, tag or commit of fmenegui/ecg-torch-renderer
DRIVE_DIR = '/content/drive/MyDrive/ecg-torch-renderer/sim2real'
ARMS = ['legacy', 'full']               # add e.g. 'full-no-phone_isp', 'full-no-report_text'
RUN_PUBLISHED_CONTROL = True            # authors' weights, for orientation
TRAIN_RECORDS, VAL_RECORDS = 1500, 150  # PTB-XL records rendered per arm (folds 1-8 / 9)
TEST_RECORDS = 40                       # real D0 records from fold 10, one per patient
KINDS = ['digital', 'color-scan', 'gray-scan', 'phone-photo']
RENDER_H, RENDER_W = 1200, 1600         # must exceed CROP in both dimensions
STEPS, BATCH, CROP = 3000, 4, 1024      # identical for every arm
SEED = 1973
RUN = time.strftime('%Y%m%d-%H%M%S')

In [ ]:
# ---- Google Drive, GPU and code --------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, json, pathlib, shutil
OUT = pathlib.Path(DRIVE_DIR) / RUN
OUT.mkdir(parents=True, exist_ok=True)
print('Results ->', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!rm -rf ecg-torch-renderer && git clone -q https://github.com/fmenegui/ecg-torch-renderer
%cd /content/ecg-torch-renderer
!git checkout -q {REPO_REF} && git log --oneline -1
# Keep Colab's preinstalled Torch; add only what the renderer and npj path need.
!pip install -q -e . yacs torch-tps scikit-image networkx scikit-learn "ray[tune]" pyyaml
!apt-get -qq install -y git-lfs > /dev/null && git lfs install --skip-repo > /dev/null
!python -m reproductions.bootstrap --dest /content/sources --method npj
UPSTREAM = '/content/sources/npj'
(OUT / 'environment.json').write_text(json.dumps({
    'repo_ref': REPO_REF,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip(),
    'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip(),
    'config': dict(ARMS=ARMS, TRAIN_RECORDS=TRAIN_RECORDS, VAL_RECORDS=VAL_RECORDS, TEST_RECORDS=TEST_RECORDS,
                   KINDS=KINDS, RENDER_H=RENDER_H, RENDER_W=RENDER_W, STEPS=STEPS, BATCH=BATCH, CROP=CROP, SEED=SEED),
}, indent=2))

In [ ]:
# ---- PTB-XL signals (training sources and test references) -----------------
# About 1.7 GB zipped. Only 500 Hz records are used.
!mkdir -p /content/data && cd /content/data && [ -d ptbxl ] || ( \
  wget -q -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip && \
  unzip -q ptbxl.zip && mv ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 ptbxl && rm ptbxl.zip )
PTBXL = '/content/data/ptbxl'
!ls {PTBXL} | head

In [ ]:
# ---- Real test images: inventory, patient-disjoint split, download ---------
!python -m reproductions.fetch_catalog --out /content/data/catalog
!python examples/sim2real_npj.py select --ptbxl-csv {PTBXL}/ptbxl_database.csv \
    --d0-files /content/data/catalog/files.json --kinds {' '.join(KINDS)} --test-records {TEST_RECORDS} \
    --train-records {TRAIN_RECORDS} --val-records {VAL_RECORDS} --seed {SEED} --out /content/data/split.json
split = json.loads(pathlib.Path('/content/data/split.json').read_text())
test_ids = [str(row['record']) for row in split['test']]
!rm -rf /content/data/real && python examples/prepare_digitization_reference.py --out /content/data/real \
    --metadata {PTBXL}/ptbxl_database.csv --kinds {' '.join(KINDS)} --records {' '.join(test_ids)}
shutil.copy('/content/data/split.json', OUT / 'split.json')
manifest = json.loads(pathlib.Path('/content/data/real/manifest.json').read_text())
print(len(manifest['cases']), 'real images from', len({c['patient_id'] for c in manifest['cases']}), 'patients')

In [ ]:
# ---- Inference and scoring helpers -----------------------------------------
REAL = pathlib.Path('/content/data/real')
images = [str(REAL / case['image']) for case in manifest['cases']]

def infer(arm, checkpoint=None):
    out = pathlib.Path('/content/runs') / arm / 'infer'
    if out.exists():
        shutil.rmtree(out)
    command = ['python', '-m', 'reproductions.npj.infer', '--upstream', UPSTREAM, '--preset', 'george-moody-2024',
               '--out', str(out)] + [arg for image in images for arg in ('--image', image)]
    if checkpoint:
        command += ['--segmentation-checkpoint', str(checkpoint)]
    # Exit code 2 only means some images raised; they are scored as missing.
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-2000:], result.stderr[-2000:])
    return out

def score(arm, predictions):
    target = OUT / arm / 'scores.json'
    !python examples/sim2real_npj.py score --manifest {REAL}/manifest.json --predictions {predictions} --arm {arm} --out {target}
    receipt = pathlib.Path(predictions) / 'receipt.json'
    if receipt.exists():
        shutil.copy(receipt, OUT / arm / 'infer-receipt.json')
    return json.loads(target.read_text())

In [ ]:
# ---- Control: the authors' published weights -------------------------------
if RUN_PUBLISHED_CONTROL:
    (OUT / 'published').mkdir(exist_ok=True)
    score('published', infer('published'))

In [ ]:
# ---- Arms: render, train with the same budget, digitize, score -------------
for arm in ARMS:
    work = pathlib.Path('/content/runs') / arm
    shutil.rmtree(work, ignore_errors=True)
    (OUT / arm).mkdir(exist_ok=True)
    !python examples/sim2real_npj.py render --split /content/data/split.json --ptbxl-root {PTBXL} \
        --recipe {arm} --out {work}/pairs --height {RENDER_H} --width {RENDER_W} --seed {SEED}
    # Keep a few rendered pairs for visual review.
    samples = OUT / arm / 'samples'
    samples.mkdir(exist_ok=True)
    for stem in ['000001', '000002', '000003', '000004']:
        for suffix in ['.png', '_mask.png']:
            shutil.copy(work / 'pairs' / 'train' / (stem + suffix), samples / (stem + suffix))
    shutil.copy(work / 'pairs' / 'render-receipt.json', OUT / arm / 'render-receipt.json')
    !python examples/sim2real_npj.py train --upstream {UPSTREAM} --train-dir {work}/pairs/train \
        --val-dir {work}/pairs/val --out {work}/train --steps {STEPS} --batch-size {BATCH} --crop {CROP} --seed {SEED}
    shutil.copy(work / 'train' / 'train-receipt.json', OUT / arm / 'train-receipt.json')
    shutil.copy(work / 'train' / 'segmentation.pt', OUT / arm / 'segmentation.pt')
    score(arm, infer(arm, work / 'train' / 'segmentation.pt'))
    shutil.rmtree(work / 'pairs')   # free disk before the next arm

In [ ]:
# ---- Summary ---------------------------------------------------------------
import math
rows = []
for arm in (['published'] if RUN_PUBLISHED_CONTROL else []) + ARMS:
    path = OUT / arm / 'scores.json'
    if not path.exists():
        continue
    for kind, value in json.loads(path.read_text())['summary'].items():
        low, high = value['patient_bootstrap_95']
        rows.append(f"| {arm} | {kind} | {value['mean_snr_db']:.2f} | {low:.2f} to {high:.2f} | "
                    f"{value['mean_printed_coverage']:.1%} | {value['returned']}/{value['images']} |")
table = ('| Arm | Capture | Mean SNR (dB) | Patient 95 % CI | Printed coverage | Returned |\n'
         '| --- | --- | ---: | --- | ---: | ---: |\n' + '\n'.join(rows))
note = (f'Run {RUN}. Same npj UNet, {STEPS} steps, batch {BATCH}, crop {CROP}, seed {SEED}, '
        f'{TRAIN_RECORDS} PTB-XL training records per arm. Test: {len(manifest["cases"])} ECG-Image-Database D0 images, '
        f'{len({c["patient_id"] for c in manifest["cases"]})} patients, none seen in training. SNR over printed '
        'samples, missing predictions scored as zero. One seed: treat differences inside the intervals as unresolved.')
(OUT / 'summary.md').write_text('# Sim-to-real benchmark\n\n' + note + '\n\n' + table + '\n')
print(note); print(); print(table)